In [12]:
!pip install -q transformers datasets torch scikit-learn

In [13]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Define our logic gate classes and sample descriptions
data = {
    "text": [
        # AND
        "output high only when both inputs are high",
        "returns 1 if all inputs are 1",
        "logical conjunction where output is true only if input a and input b are true",
        "both inputs must be high for output to be 1",
        "high output exclusively when both inputs are active",

        # OR
        "output is high if at least one input is high",
        "returns 1 if either input a or input b is 1",
        "logical disjunction giving true if any input is true",
        "active output when one or more inputs are high",
        "positive output if at least one condition is met",

        # NOT
        "inverts the input signal",
        "output is the exact opposite of the input",
        "high input gives low output and vice versa",
        "logical negation inverter",
        "complement of the single input",

        # NAND
        "not and gate output is low only when both inputs are high otherwise high",
        "inverted and gate",
        "high unless all inputs are high",
        "output is 0 only when both a and b are 1",

        # NOR
        "not or gate output is high only when all inputs are low",
        "inverted or gate",
        "returns 1 only if neither input is high",
        "low output if any input is high",

        # XOR
        "exclusive or output is high when inputs are different",
        "returns 1 if inputs mismatch",
        "high when either a or b is high but not both",
        "odd number of high inputs gives true",

        # XNOR
        "exclusive nor output is high when inputs are identical",
        "returns 1 if inputs match",
        "high when both inputs are same either both 0 or both 1",
        "equivalence gate"
    ],
    "label": [
        0, 0, 0, 0, 0,  # AND
        1, 1, 1, 1, 1,  # OR
        2, 2, 2, 2, 2,  # NOT
        3, 3, 3, 3,     # NAND
        4, 4, 4, 4,     # NOR
        5, 5, 5, 5,     # XOR
        6, 6, 6, 6      # XNOR
    ]
}

id2label = {0: "AND", 1: "OR", 2: "NOT", 3: "NAND", 4: "NOR", 5: "XOR", 6: "XNOR"}
label2id = {"AND": 0, "OR": 1, "NOT": 2, "NAND": 3, "NOR": 4, "XOR": 5, "XNOR": 6}

# Create Hugging Face Dataset
dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

In [14]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # Removed tokenizer=tokenizer from here!
)

# Train the model
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,2.023680
2,1.927069,2.035005
3,1.927069,2.045359
4,1.911059,2.051998
5,1.873666,2.053435


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=15, training_loss=1.903931490580241, metrics={'train_runtime': 138.0552, 'train_samples_per_second': 0.869, 'train_steps_per_second': 0.109, 'total_flos': 1987188157440.0, 'train_loss': 1.903931490580241, 'epoch': 5.0})

In [16]:
def predict_gate(description):
    device = model.device
    inputs = tokenizer(description, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    predicted_class_id = torch.argmax(logits, dim=-1).item()
    confidence = torch.softmax(logits, dim=-1)[0][predicted_class_id].item()

    gate_name = id2label[predicted_class_id]
    return {"gate": gate_name, "confidence": round(confidence * 100, 2)}

# Test the model with natural language prompts
test_queries = [
    "gives a high signal only when both inputs are 1",
    "output is zero if any input is high",
    "reverses the incoming bit"
]

for query in test_queries:
    result = predict_gate(query)
    print(f"Query: '{query}'\n--> Predicted Gate: {result['gate']} ({result['confidence']}% confidence)\n")

Query: 'gives a high signal only when both inputs are 1'
--> Predicted Gate: AND (16.76% confidence)

Query: 'output is zero if any input is high'
--> Predicted Gate: AND (16.51% confidence)

Query: 'reverses the incoming bit'
--> Predicted Gate: AND (15.87% confidence)



In [18]:
# Save the trained model and tokenizer locally or export to your drive
model.save_pretrained("./logic_gate_model")
tokenizer.save_pretrained("./logic_gate_model")

# If you want to zip it and download it to use in your backend:
!zip -r logic_gate_model.zip logic_gate_model/
from google.colab import files
files.download("logic_gate_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

updating: logic_gate_model/ (stored 0%)
updating: logic_gate_model/config.json (deflated 53%)
updating: logic_gate_model/tokenizer.json (deflated 71%)
updating: logic_gate_model/model.safetensors (deflated 8%)
updating: logic_gate_model/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>